# Extended imputation benchmark — *everything else* (Colab Pro)

Runs the **20 imputation techniques** the main benchmark had left out (no public
reference implementation / bespoke pipelines), scored by the **same** reconstruction
protocol so they merge straight into the 40-method leaderboard.

**What it does**
- `git clone`s the repo (all processed data is committed — no data prep needed).
- Installs only `tensorly` (torch is already on Colab; nothing else is needed).
- Runs `src.imputation_benchmark_extended.run_extended_resumable` over Dhaka / Beijing / Delhi.
- **Saves every method to Google Drive the moment it finishes and resumes after a disconnect** —
  just re-run the *Run* cell.

**Honest notes.** Three techniques stay impossible (Kriging, Cold deck — missing data; exact
GA/FOA search — replaced by defaults). Several methods here are *representative approximations*
of a family, not line-for-line paper reproductions — see `extended_coverage_map()`. A GPU only
speeds up the 2 deep methods (`convlstm`, `cnn_gan`); the other 18 are CPU-bound and run at the
same speed either way.

## 1 · Clone the repo

In [ ]:
import os, sys
REPO = "Missingness-Aware-Transformer-for-Air-Quality-Forecasting"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Nafis878/{REPO}.git
%cd {REPO}/air-transformer
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 2 · Install the one extra dependency (don't touch Colab's torch)

In [ ]:
!pip install -q tensorly
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 3 · Mount Drive + config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- knobs ----
SMOKE        = True          # True: 2 stations, MCAR only, ~2 min wiring check. Set False for the full run.
INCLUDE_DEEP = True          # convlstm + cnn_gan (GPU recommended)
RESULTS_DIR  = '/content/drive/MyDrive/imputation_benchmark'   # same folder as the base run
DATASETS = [('dhaka',   'config.yaml'),
            ('beijing', 'config_beijing.yaml'),
            ('delhi',   'config_delhi.yaml')]

RUN_DIR = RESULTS_DIR + ('_smoke' if SMOKE else '')   # smoke writes elsewhere so it can't poison the real run
print("writing to:", RUN_DIR, "| SMOKE =", SMOKE, "| deep =", INCLUDE_DEEP)

## 4 · Run (resumable)

Re-run this cell after any disconnect — finished methods are skipped, only the one in flight repeats.

In [ ]:
import json, yaml, warnings, os
warnings.filterwarnings('ignore')
from src import imputation_benchmark_extended as ext

extra = dict(patterns=('mcar',), max_stations=2, max_slice_len=800) if SMOKE else {}
for name, cfgp in DATASETS:
    if not os.path.exists(cfgp):
        print(f"[{name}] {cfgp} missing -> skip"); continue
    cfg = yaml.safe_load(open(cfgp)); cfg['dataset_name'] = name
    proc = cfg['paths']['processed_dir']
    parq = os.path.join(proc, 'all_stations.parquet')
    if not os.path.exists(parq):
        print(f"[{name}] {parq} missing -> skip"); continue
    sc = json.load(open(os.path.join(proc, 'scalers.json')))
    print(f"
===== {name.upper()} =====")
    ext.run_extended_resumable(cfg, sc, RUN_DIR, seed=42,
                               include_deep=INCLUDE_DEEP, device=DEVICE,
                               log=print, **extra)
print("
DONE.")

## 5 · Aggregate + merge with the base 40-method leaderboard

In [ ]:
import pandas as pd
from src.imputation_benchmark import aggregate_all

ext_board = aggregate_all(RUN_DIR)        # extended methods (reads the per_method CSVs on Drive)
ext_board.to_csv(os.path.join(RUN_DIR, 'leaderboard_extended.csv'), index=False)
print("extended methods:", ext_board.method.nunique(), "| rows:", len(ext_board))

# combine with the committed full run for one 60-method table
base_path = 'outputs/imputation_benchmark/leaderboard_all.csv'
if os.path.exists(base_path):
    base = pd.read_csv(base_path)
    combined = (pd.concat([base, ext_board], ignore_index=True)
                  .drop_duplicates(['dataset', 'method', 'pattern'], keep='last'))
    combined.to_csv(os.path.join(RUN_DIR, 'leaderboard_all_combined.csv'), index=False)
    print("combined methods:", combined.method.nunique(), "| rows:", len(combined))
else:
    combined = ext_board
combined.head()

## 6 · Updated coverage map (what is now run vs still impossible)

In [ ]:
from src.imputation_benchmark import coverage_map
from src.imputation_benchmark_extended import extended_coverage_map

cov = coverage_map().set_index('technique')
upd = extended_coverage_map().set_index('technique')
cov.update(upd)                                   # promote the newly-run techniques
cov = cov.reset_index()
cov.to_csv(os.path.join(RUN_DIR, 'coverage_map_updated.csv'), index=False)
print(cov['status'].value_counts().to_dict())
cov[cov.status == 'skipped']

## 7 · Leaderboards (top methods per dataset × pattern)

In [ ]:
for ds in combined.dataset.unique():
    for pat in combined.pattern.unique():
        sub = combined[(combined.dataset == ds) & (combined.pattern == pat)]
        if sub.empty: continue
        sub = sub.sort_values('imputability', ascending=False).head(8)
        print(f"
### {ds} / {pat}")
        print(sub[['method','family','imputability','pm25_rmse_ugm3']].to_string(index=False))

## 8 · Figures (CSDI + any divergent outlier excluded for readability)

In [ ]:
import sys; sys.path.insert(0, 'scripts')
from run_imputation_benchmark import _figures
figdir = os.path.join(RUN_DIR)
clean = combined[(combined.method != 'csdi') & (combined.overall_std_rmse.abs() <= 5)].copy()
_figures(clean, figdir)
from IPython.display import Image, display
for f in ['imputability_heatmap.png']:
    p = os.path.join(figdir, 'figures', f)
    if os.path.exists(p): display(Image(p))

## 9 · Zip the results and download

In [ ]:
import shutil
from google.colab import files
z = shutil.make_archive('/content/imputation_extended_artifacts', 'zip', RUN_DIR)
print("wrote", z)
files.download(z)